# Projekt 01 (basic) — Punktoperationen & Histogrammausgleich

**Modul 12 — Image Processing** · Format: **Jupyter Notebook** (`punktops_histogramm.ipynb`)

Die einfachsten Bildoperationen sind **Punktoperationen**: jeder Pixel wird *unabhängig*
von seinen Nachbarn umgerechnet. Trotz ihrer Einfachheit stecken sie hinter Helligkeit,
Kontrast, Gamma und dem **Histogrammausgleich** — einer der schönsten kleinen Algorithmen
der Bildverarbeitung. Du baust sie hier von Hand und machst ihre Wirkung am Histogramm
sichtbar.

Kernfragen (Skript-Abschnitt 1):
- Wie verändern lineare Punktoperationen und **Gamma** ein Bild?
- Was beschreibt das **Histogramm**, und wie streckt der **Ausgleich über die CDF** den
  Kontrast?

> **Viel Anleitung, kein Training, nur `numpy`.** Läuft in Sekunden.


## Setup

Benötigt `numpy`, `matplotlib`, `Pillow` (Repo-`requirements.txt`). Das Beispielbild
(*Grace Hopper*) ist in matplotlib enthalten — kein Download.

```bash
source ../../../../.venv/bin/activate
jupyter lab      # oder das Notebook in VS Code öffnen, Kernel = Repo-.venv
```


## Teil A — Bild & Histogramm *(vorgegeben)*

Wir laden ein Graustufenbild (Werte $0$–$255$) und zeigen es mit seinem **Histogramm** —
der Verteilung der Grauwerte.


In [ ]:
# ---- Bild + Histogramm (vorgegeben) --------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
from PIL import Image

with cbook.get_sample_data("grace_hopper.jpg") as f:
    img = np.asarray(Image.open(f).convert("L"), dtype=np.uint8)   # Graustufen 0..255
print("Bild:", img.shape, "dtype", img.dtype, "min/max", img.min(), img.max())

def show_with_hist(images, titles):
    n = len(images)
    fig, ax = plt.subplots(2, n, figsize=(4 * n, 6))
    ax = np.array(ax).reshape(2, n)          # auch bei n=1 eine 2D-Achsenmatrix
    for k, (im, t) in enumerate(zip(images, titles)):
        ax[0, k].imshow(im, cmap="gray", vmin=0, vmax=255); ax[0, k].set_title(t); ax[0, k].axis("off")
        ax[1, k].hist(im.ravel(), bins=256, range=(0, 255), color="steelblue")
        ax[1, k].set_xlim(0, 255); ax[1, k].set_yticks([])
    plt.tight_layout(); plt.show()

show_with_hist([img], ["Original"])


### Aufgabe 1 — Lineare Punktoperation & Gamma

Implementiere zwei Punktoperationen $s=T(r)$ (Skript 1.2). Achte auf **Clipping** auf
$[0,255]$ und `uint8`.

- **Linear** (Kontrast $a$, Helligkeit $b$): $s = a\,r + b$.
- **Gamma:** $s = 255\,(r/255)^{\gamma}$ — $\gamma<1$ hellt dunkle Bereiche auf.

**Deine Aufgabe (`# TODO`):** fülle `linear(img, a, b)` und `gamma(img, g)`.


In [ ]:
# ---- AUFGABE 1: Punktoperationen -----------------------------------------
def linear(im, a, b):
    # TODO: s = a*r + b, auf [0,255] clippen, als uint8 zurückgeben
    raise NotImplementedError("Aufgabe 1a: linear implementieren")

def gamma(im, g):
    # TODO: s = 255*(r/255)**g, clippen, uint8
    raise NotImplementedError("Aufgabe 1b: gamma implementieren")

dark = linear(img, 1.0, -60)
contr = linear(img, 1.6, -60)
g_bright = gamma(img, 0.5)
show_with_hist([img, dark, contr, g_bright],
               ["Original", "Heller/Dunkler (b=-60)", "Kontrast (a=1.6)", "Gamma 0.5"])


### Aufgabe 2 — Histogrammausgleich von Hand

Der **Histogrammausgleich** nutzt die **kumulative Verteilung** (CDF) als Abbildung
(Skript 1.3):
$$s = T(r) = \operatorname{round}\!\big((L-1)\cdot \text{cdf}(r)\big),\quad L=256.$$

**Deine Aufgabe (`# TODO`):** implementiere `equalize(img)`:
1. Histogramm der 256 Grauwerte (`np.bincount(im.ravel(), minlength=256)`);
2. normieren zu $p(k)$, kumulieren zur `cdf`;
3. Abbildung `T = round(255 * cdf)` (als `uint8`), dann `T[img]` anwenden.


In [ ]:
# ---- AUFGABE 2: Histogrammausgleich --------------------------------------
def equalize(im):
    # TODO: Histogramm -> cdf -> T=round(255*cdf) (uint8) -> return T[im]
    raise NotImplementedError("Aufgabe 2: equalize implementieren")

low_contrast = linear(img, 0.4, 90)
equalized = equalize(low_contrast)
show_with_hist([low_contrast, equalized, equalize(img)],
               ["Kontrastarm", "Ausgeglichen", "Ausgleich(Original)"])


### Aufgabe 3 — CDF sichtbar machen & Selbstcheck

**Deine Aufgabe (`# TODO`):** Plotte die **CDF** des kontrastarmen Bildes vor und nach dem
Ausgleich. Nach dem Ausgleich sollte die CDF **annähernd eine Gerade** (Diagonale) sein —
das ist die grafische Signatur der Gleichverteilung.


In [ ]:
# ---- AUFGABE 3: CDF vor/nach Ausgleich -----------------------------------
def cdf_of(im):
    h = np.bincount(im.ravel(), minlength=256).astype(np.float64)
    return np.cumsum(h) / h.sum()

# TODO: cdf_of(low_contrast) und cdf_of(equalized) plotten, plus die Ideal-Diagonale
raise NotImplementedError("Aufgabe 3: CDF vor/nach Ausgleich plotten")

print("Wertebereich kontrastarm:", low_contrast.min(), "-", low_contrast.max())
print("Wertebereich ausgeglichen:", equalized.min(), "-", equalized.max())


## Reflexion (kurz, schriftlich)

1. **Gamma:** Warum hellt $\gamma<1$ *dunkle* Bereiche stärker auf als helle? (Denk an den
   Verlauf von $x^\gamma$ auf $[0,1]$.)
2. **Ausgleich:** Warum macht die CDF-Abbildung das Histogramm (annähernd) gleichverteilt,
   und warum ist die CDF danach fast eine Gerade?
3. **Grenzen:** Histogrammausgleich ist *global*. Bei einem Bild mit sehr hellen und sehr
   dunklen Regionen kann er lokal übersteuern. Welche Idee (Stichwort *adaptiv*, z. B.
   CLAHE) würde das lokal begrenzen?
4. **Punktoperationen:** Warum kann *keine* Punktoperation Rauschen entfernen oder Kanten
   schärfen? (Was bräuchte man stattdessen — Ausblick auf Projekt 02?)

> **Referenz:** Nach dem Ausgleich nutzt das Bild (fast) den vollen Bereich 0–255, das
> Histogramm ist breiter/flacher, die CDF nahe der Diagonale. Gamma 0.5 hebt sichtbar
> Schattendetails.
